In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Gestión del rendimiento: una función de Qiskit de Q-CTRL Fire Opal
*Consulta la [referencia de la API](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** Las funciones de Qiskit son una funcionalidad experimental disponible únicamente para los usuarios de los planes IBM Quantum&reg; Premium, Flex y On-Prem (a través de la API de IBM Quantum Platform). Se encuentran en estado de versión preliminar y están sujetas a cambios.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>
## Descripción general
Fire Opal Performance Management hace que sea sencillo para cualquier persona obtener resultados significativos de las computadoras cuánticas a escala sin necesidad de ser experto en hardware cuántico. Al ejecutar circuitos con Fire Opal Performance Management, se aplican automáticamente técnicas de supresión de errores basadas en inteligencia artificial, lo que permite escalar problemas más grandes con más puertas y qubits. Este enfoque reduce la cantidad de shots necesarios para obtener la respuesta correcta, sin sobrecarga adicional, lo que resulta en un ahorro significativo tanto en tiempo de cómputo como en costos.

Performance Management suprime los errores y aumenta la probabilidad de obtener la respuesta correcta en hardware con ruido. En otras palabras, aumenta la relación señal-ruido. La siguiente imagen muestra cómo la mayor precisión habilitada por Performance Management puede reducir la necesidad de shots adicionales en el caso de un algoritmo de Transformada de Fourier Cuántica de 10 qubits. Con solo 30 shots, Q-CTRL alcanza el umbral de confianza del 99 %, mientras que el método predeterminado (`QiskitRuntime` Sampler, `optimization_level`=3 y `resilience_level`=1, `ibm_sherbrooke`) requiere 170 000 shots. Al obtener la respuesta correcta más rápido, ahorras un tiempo de cómputo significativo.

![Visualización del tiempo de ejecución mejorado](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

La función Performance Management se puede usar con cualquier algoritmo, y puedes utilizarla fácilmente en lugar de las [primitivas estándar de Qiskit Runtime](/guides/primitives). Internamente, múltiples técnicas de supresión de errores trabajan juntas para evitar que los errores ocurran en tiempo de ejecución. Todos los métodos del pipeline de Fire Opal están preconfigurados y son agnósticos al algoritmo, lo que significa que siempre obtienes el mejor rendimiento desde el primer momento.

Para obtener acceso a Performance Management, [contacta a Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Descripción
Fire Opal Performance Management ofrece dos opciones de ejecución similares a las primitivas de Qiskit Runtime, por lo que puedes intercambiarlas fácilmente con el Sampler y el Estimator de Q-CTRL. El flujo de trabajo general para usar la función Performance Management es:
1. Define tu circuito (y operadores en el caso del Estimator).
2. Ejecuta el circuito.
3. Recupera los resultados.

Para reducir el ruido del hardware, Fire Opal emplea una serie de técnicas de supresión de errores basadas en inteligencia artificial que se muestran en la siguiente imagen. Con Fire Opal, todo el pipeline está completamente automatizado sin necesidad de ninguna configuración.

El pipeline de Fire Opal elimina la necesidad de sobrecargas adicionales, como un mayor tiempo de ejecución cuántica o qubits físicos adicionales. Cabe señalar que el tiempo de procesamiento clásico sigue siendo un factor (consulta la sección de [Benchmarks](#benchmarks) para obtener estimaciones, donde "Tiempo total" refleja el procesamiento tanto clásico como cuántico). A diferencia de la mitigación de errores, que requiere sobrecarga en forma de muestreo, la supresión de errores de Fire Opal funciona tanto a nivel de puertas como de pulsos para abordar diversas fuentes de ruido y prevenir la probabilidad de que ocurra un error. Al prevenir los errores, se elimina la necesidad de un posprocesamiento costoso.

La siguiente imagen muestra los métodos de supresión de errores automatizados por Fire Opal Performance Management.

![Visualización del pipeline de supresión de errores](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

La función ofrece dos primitivas, Sampler y Estimator, y las entradas y salidas de ambas extienden la especificación implementada para las [primitivas V2 de Qiskit Runtime](/guides/primitive-input-output#pubs).
## Benchmarks
Los resultados de [benchmarking algorítmico publicados](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) demuestran una mejora significativa del rendimiento en diversos algoritmos, incluidos Bernstein-Vazirani, la transformada de Fourier cuántica, la búsqueda de Grover, el algoritmo de optimización aproximada cuántica y el estimador variacional de valores propios cuánticos. El resto de esta sección ofrece más detalles sobre los tipos de algoritmos que puedes ejecutar, así como el rendimiento y los tiempos de ejecución esperados.

Los siguientes estudios independientes demuestran cómo el Performance Management de Q-CTRL permite la investigación algorítmica a escala sin precedentes:
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) - aprendizaje de kernels cuánticos de hasta 50 qubits
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) - estimación de fase cuántica de hasta 33 qubits
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) - carga de datos cuánticos de hasta 21 qubits

La siguiente tabla ofrece una guía aproximada sobre la precisión y los tiempos de ejecución de benchmarks previos realizados en `ibm_fez`. El rendimiento en otros dispositivos puede variar. El tiempo de uso se basa en un supuesto de 10 000 shots por circuito. El "Número de qubits" indicado no es un límite estricto, sino que representa umbrales aproximados donde puedes esperar una precisión de solución extremadamente consistente. Se han resuelto con éxito problemas de mayor tamaño, y se recomienda probar más allá de estos límites.

| Ejemplo    | Número de qubits | Precisión | Medida de precisión | Tiempo total (s) | Uso de runtime (s) | Primitiva (Modo) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100%  | Tasa de éxito (porcentaje de ejecuciones donde la respuesta correcta es la cadena de bits con mayor conteo)     | 10    | 8         | Sampler |
| Transformada de Fourier Cuántica | 30Q              | 100% | Tasa de éxito (porcentaje de ejecuciones donde la respuesta correcta es la cadena de bits con mayor conteo)      | 10    | 8        | Sampler |
| Estimación de Fase Cuántica  | 30Q   | 99.9998%  | Precisión del ángulo encontrado: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| Simulación cuántica: modelo de Ising (15 pasos) | 20Q  | 99.775%   |  $A$ (definido a continuación)  |  60 (por paso)  | 15 (por paso)   | Estimator |
| Simulación cuántica 2: dinámica molecular (20 puntos temporales) | 34Q  |  96.78%  |  $A_{mean}$ (definido a continuación)   | 10 (por punto temporal)    | 6 (por punto temporal)  | Estimator |

Definiendo la precisión de la medición de un valor esperado, la métrica $A$ se define de la siguiente manera:
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
donde $ \epsilon^{ideal} $ = valor esperado ideal, $ \epsilon^{meas} $ = valor esperado medido, $\epsilon^{ideal}_{max} $ = valor máximo ideal, y $\epsilon^{ideal}_{min}$ = valor mínimo ideal. $A_{mean}$ es simplemente el promedio del valor de $A$ a través de múltiples mediciones.

Esta métrica se utiliza porque es invariante a desplazamientos globales y escalados en el rango de valores alcanzables. En otras palabras, independientemente de si desplazas el rango de posibles valores esperados hacia arriba o hacia abajo, o aumentas la dispersión, el valor de $A$ debe permanecer consistente.
## Primeros pasos
Fire Opal Performance Management utiliza Qiskit v`2.0.0`, que es la versión recomendada. Las versiones compatibles son Qiskit >=v`2.0.0`.
Autentícate usando tu [clave API de IBM Quantum Platform](http://quantum.cloud.ibm.com/) y selecciona la función de Qiskit de la siguiente manera. (Este fragmento asume que ya has [guardado tu cuenta](/guides/functions#install-qiskit-functions-catalog-client) en tu entorno local.)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. Ejecutar el circuito**

Ejecuta el circuito y, de manera opcional, define el backend y el número de shots.

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

Puedes usar las conocidas [APIs de Qiskit Serverless](/guides/serverless) para comprobar el estado de tu carga de trabajo de la función de Qiskit:

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. Recuperar el resultado**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

Los resultados tienen el mismo formato que un [resultado de Estimator](/guides/estimator-input-output):

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Primitiva Sampler
### Ejemplo de Sampler
Usa la primitiva Sampler de Fire Opal Performance Management para ejecutar un circuito de Bernstein–Vazirani. Este algoritmo, utilizado para encontrar una cadena oculta a partir de las salidas de una función de caja negra, es un algoritmo de benchmarking común porque tiene una única respuesta correcta.
**1. Crear el circuito**

Define la respuesta correcta al algoritmo, la cadena de bits oculta, y el circuito de Bernstein–Vazirani. Puedes ajustar el ancho del circuito simplemente cambiando `circuit_width`.

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. Ejecutar el circuito**

Ejecuta el circuito y, de manera opcional, define el backend y el número de shots.

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

Comprueba el [estado](/guides/functions#check-job-status) de tu carga de trabajo de la función de Qiskit o recupera los [resultados](/guides/functions#retrieve-results) de la siguiente manera:

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. Graficar las principales cadenas de bits**

Grafica la cadena de bits con el mayor número de conteos para ver si la cadena de bits oculta fue la moda.